# Clon Docente en GPU gratuita de Kaggle

Ejecuta el software completo en una GPU de Kaggle (sin costo) y lo publica en una URL HTTPS temporal.

**Antes de ejecutar** (panel derecho → *Settings*):
1. **Accelerator:** `GPU T4 x2` o `GPU P100`.
2. **Internet:** *On* (Kaggle exige verificar la cuenta con un número de teléfono).

Luego: *Run All*. Al final aparecerá un enlace `https://…trycloudflare.com`. Ábralo en el navegador (también funciona en el celular).

> La sesión es temporal: **descargue sus videos** antes de cerrarla (última celda). Motor: Wav2Lip acelerado por GPU.

In [ ]:
# ===== Configuración =====
REPO_URL  = "https://github.com/PGBeermann/myClone.git"   # repositorio con el código
APP_TOKEN = "cambie-esta-clave"                                  # clave para entrar a la interfaz (URL pública)

In [ ]:
import subprocess, os
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/myClone && git clone -q --depth 1 $REPO_URL /kaggle/working/myClone
!pip install -q fastapi "uvicorn[standard]" python-multipart pyyaml
!cd /kaggle/working/myClone && WAV2LIP_DIR=/kaggle/working/Wav2Lip SKIP_PIP=1 bash install/install_wav2lip.sh

In [ ]:
env = dict(os.environ, ENGINE="wav2lip", WAV2LIP_DIR="/kaggle/working/Wav2Lip", WAV2LIP_PYTHON="python",
           DATA_DIR="/kaggle/working/data", APP_TOKEN=APP_TOKEN, W2L_BATCH="128")
server = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
                          cwd="/kaggle/working/myClone", env=env,
                          stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)
import time, urllib.request; time.sleep(5)
print(urllib.request.urlopen("http://127.0.0.1:8000/api/status").read().decode())

In [ ]:
# Túnel HTTPS gratuito (Cloudflare Quick Tunnel, sin cuenta)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared
import re
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n✅ Abra la aplicación en:", m.group(0), "\n   Clave de acceso:", APP_TOKEN); break

Mantenga el cuaderno abierto mientras usa la aplicación. Para ver la bitácora del servidor: `!tail -50 /kaggle/working/server.log`

In [ ]:
# Al terminar: empaquetar los videos generados para descargarlos (panel Output → clon_videos.zip)
!cd /kaggle/working/data && zip -qr /kaggle/working/clon_videos.zip outputs && ls -lh /kaggle/working/clon_videos.zip